## Fase Final: Generación del Corpus de Preguntas Maestras (K=12)

El objetivo de este cuaderno es realizar la generación final de las Preguntas Maestras utilizando la arquitectura validada de **12 clústeres** sobre el corpus de prospectos con **deduplicación semántica (umbral 0.95)**. 

En esta fase, se despliega un comité de **5 modelos de lenguaje de gran capacidad** para garantizar la máxima convergencia semántica y eliminar cualquier sesgo individual de los modelos. El resultado final es el artefacto `preguntas_originales_v4_5modelos.csv`, que sirve de base para la evaluación final del sistema.

### 5.1. Carga del Corpus Refinado
Utilizamos el dataset `prospectos_unicos_semanticos_15.csv`, el cual ha pasado por un proceso de filtrado TF-IDF para eliminar la redundancia estructural de la AEMPS. Esto asegura que el clustering trabaje sobre "unidades de información únicas", permitiendo que los 12 tópicos resultantes sean representativos de categorías universales y no de repeticiones burocráticas.

In [1]:
import pandas as pd
df_parrafos = pd.read_csv("prospectos_unicos_semanticos_15.csv")

df_parrafos = df_parrafos.drop_duplicates(subset=['parrafo_anonimizado'])

In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

# Cargar modelo de embeddings (multilingüe para español)
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# 1. Limpiar y vectorizar
textos = df_parrafos['parrafo_anonimizado'].tolist()
embeddings = model.encode(textos, show_progress_bar=True)

# 2. Clustering
num_clusters = 12 
clustering_model = KMeans(n_clusters=num_clusters, random_state=42)
df_parrafos['cluster'] = clustering_model.fit_predict(embeddings)

c:\Users\04jul\Desktop\CDIA\TFG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 282.99it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 869/869 [13:07<00:00,  1.10it/s]


In [3]:
df_parrafos.to_csv("prospectos_clusters12.csv", index=False)

In [4]:
import pandas as pd
df_parrafos = pd.read_csv("prospectos_clusters12.csv")

In [5]:
df_parrafos[df_parrafos["cluster"]==11]["parrafo_anonimizado"]

59       Fecha de la última revisión de este prospecto:...
166      Fecha de la última revisión de este prospecto:...
225      Fecha de la última revisión de este prospecto:...
345      Fecha de la última revisión de este prospecto:...
480      Fecha de la última revisión de este prospecto:...
                               ...                        
25935    Fecha de la última revisión de este prospecto:...
26150    Fechas de la última revisión de este prospecto...
26425    Fecha de la última revisión de este prospecto:...
26631    Este prospecto ha sido revisado en Septiembre ...
26649    Fecha de la última revisión de este prospecto:...
Name: parrafo_anonimizado, Length: 163, dtype: str

## Preparación de prompt

In [6]:
def generar_prompt_final(cluster_id, ejemplos):
  ejemplos_str = "\n- ".join(ejemplos)
      
      # PROMPT REFINADO: Foco absoluto en el usuario final
  prompt = f"""[INST] <<SYS>>
  Eres un PACIENTE que tiene una duda urgente y busca la respuesta en estos fragmentos. 
  TU REGLA MÁXIMA: Escribe ÚNICAMENTE la pregunta que harías a tu médico.
  PROHIBIDO usar palabras como: 'texto', 'fragmento', 'prospecto', 'diseño', 'JSON', 'información', 'sección', 'analizar'.
  PROHIBIDO mencionar nombres propios de medicamentos.
  <</SYS>>
  Sigue este razonamiento interno:
  1. Este fragmento "Puede disminuir la capacidad para conducir o manejar maquinaria por contener propilenglicol, ya que puede producir síntomas parecidos a los del alcohol." responde a la pregunta "¿Puedo conducir después de tomar [MEDICAMENTO]?".
  2. Este fragmento "Estas dosis se pueden repetir cada 4 horas." no responde a esa pregunta, responde a "¿Cada cuánto tiempo me tengo que tomar [MEDICAMENTO]?".
  3. Este fragmento Los demás componentes son celulosa microcristalina (E460), carboximetilalmidón sódico (Tipo A) de patata, povidona K90 (E1201)" responde a "¿Cuál es la composición de este medicamento?"
  4. Teniendo esto en cuenta, ¿qué PREGUNTA UNIVERSAL responden todos los fragmentos del grupo?
  Basándote en estos fragmentos:
  {ejemplos_str}

  Identifica el tema que se repite en el 80% de los fragmentos y descarta los detalles que solo aparecen en uno solo y genera la PREGUNTA DIRECTA que un paciente normal (no experto) haría y a la que respondan esos fragmentos. 

  Responde exclusivamente en este formato JSON:
  {{
    "pregunta_del_paciente": "Escribe aquí la pregunta"
  }}
  [/INST]"""
  return prompt

## Conexión con Ollama

In [7]:
import requests
import json
import pandas as pd

def consultar_ollama_stream(prompt, modelo_id):
    url = "https://wiig.dia.fi.upm.es/ollama/v1/chat/completions"
    payload = {
        "model": modelo_id,
        "messages": [{"role": "user", "content": prompt}],
        "stream": True
    }
    
    texto_acumulado = ""
    try:
        with requests.post(url, json=payload, stream=True, timeout=1000) as r:
            r.raise_for_status()
            for line in r.iter_lines():
                if not line: continue
                decoded = line.decode("utf-8").replace("data: ", "").strip()
                if decoded == "[DONE]": break
                try:
                    chunk = json.loads(decoded)
                    content = chunk["choices"][0].get("delta", {}).get("content", "")
                    texto_acumulado += content
                except: continue
        return texto_acumulado
    except Exception as e:
        return f"ERROR: {str(e)}"

### 5.2. Inferencia Generativa mediante Comité de Expertos (Pentapartito)
Para mitigar alucinaciones y sesgos léxicos detectados en fases anteriores (como el uso de idiomas extranjeros o términos excesivamente técnicos, implementamos un proceso de consulta a **cinco modelos SOTA** de gran escala:
1. **DeepSeek-R1 (32B):** Especializado en razonamiento lógico y extracción.
2. **Gemma-3 (27B):** Modelo con alta capacidad de abstracción y seguimiento de instrucciones.
3. **GPT-OSS (120B):** Máxima capacidad de parámetros para asegurar coherencia gramatical.
4. **Nemotron-3-Nano (30B):** Modelo optimizado para tareas de recuperación y síntesis.
5. **Qwen3 (30B):** (Integrado en fases de control).

**Estrategia de Abstracción:** Aplicamos el *Prompt* de Arquitectura de Información Médica, prohibiendo términos específicos y forzando la creación de **Preguntas de Nivel Superior** que actúen como categorías universales válidas para cualquier fármaco.

In [9]:
anotadores = ["qwen3:30b", "deepseek-r1:32b", "gemma3:27b", "gpt-oss:120b", "nemotron-3-nano:30b"]
resultados = []

In [10]:
import time

for c_id in sorted(df_parrafos['cluster'].unique()):
    print(f" Procesando Cluster {c_id}...")
    
    # 1. Selección de variedad
    df_c = df_parrafos[df_parrafos['cluster'] == c_id]
    ejemplos_variados = df_c["parrafo_anonimizado"].to_list()
    ejemplos_cortos = [texto[:500] + "..." if len(texto) > 500 else texto for texto in ejemplos_variados]
    
    # 2. Generación del prompt
    prompt_texto = generar_prompt_final(c_id, ejemplos_cortos)
    
    fila = {"cluster": c_id, "ejemplos_usados": "|".join(ejemplos_cortos)}
    
    # 3. Consulta a los LLMs
    for modelo in anotadores:
        print(f"  -> Consultando {modelo}...")
        respuesta = consultar_ollama_stream(prompt_texto, modelo)
        print(respuesta)
        fila[f"res_{modelo.replace(':', '_')}"] = respuesta
        time.sleep(1) # Pausa para no saturar el servidor

    resultados.append(fila)

df_final = pd.DataFrame(resultados)
df_final.to_csv("resultados_etiquetado_12_topicos_5modelos.csv", index=False)

 Procesando Cluster 0...
  -> Consultando qwen3:30b...
{
  "pregunta_del_paciente": "¿Este medicamento es exento de sodio?"
}
  -> Consultando deepseek-r1:32b...
```json
{
  "pregunta_del_paciente": "¿Cuál es la dosis correcta para este medicamento y cómo se toma?"
}
```
  -> Consultando gemma3:27b...
```json
{
  "pregunta_del_paciente": "¿Cómo debo tomar este medicamento y cada cuánto tiempo?"
}
```
  -> Consultando gpt-oss:120b...
{
  "pregunta_del_paciente": "¿Cuál es la dosis recomendada de este medicamento?"
}
  -> Consultando nemotron-3-nano:30b...
{
  "pregunta_del_paciente": "¿Cuál es la dosis y la frecuencia de administración?"
}
 Procesando Cluster 1...
  -> Consultando qwen3:30b...
{
  "pregunta_del_paciente": "¿Puedo tomar este medicamento si estoy embarazada o en periodo de lactancia?"
}
  -> Consultando deepseek-r1:32b...
{
    "pregunta_del_paciente": "¿Cuáles son las advertencias que deben tener las mujeres embarazadas y en período de lactancia al momento de usar medica

### 5.3. Post-Procesamiento y Curación de Datos
Dada la diversidad de formatos de salida de los modelos (especialmente el uso de etiquetas de pensamiento) aplicamos una limpieza para conseguir un único archivo con las 60 preguntas (12 por cada uno de los 5 modelos)

In [ ]:
import pandas as pd
import json
import re

def limpiar_y_extraer_json_robusto(texto):
    """
    Extrae la pregunta del paciente ignorando texto explicativo, 
    bloques Markdown y errores de servidor.
    """
    if pd.isna(texto) or str(texto).strip() == "":
        return "N/A"
    
    texto_str = str(texto).strip()
    
    
    if "Error" in texto_str or "ERROR" in texto_str:
        return "ERROR_SERVIDOR"

    match = re.search(r'\{.*\}', texto_str, re.DOTALL)
    
    if match:
        json_blob = match.group()
        try:
            # Intentar cargar como JSON estándar
            data = json.loads(json_blob)
            if isinstance(data, dict):
                # Buscamos las llaves más probables
                for llave in ["pregunta_del_paciente", "pregunta", "question"]:
                    if llave in data:
                        return data[llave]
                # Si es un diccionario pero no tiene esas llaves, devolvemos el primer valor que sea texto
                for valor in data.values():
                    if isinstance(valor, str): return valor
            return json_blob
        except:
            # Si el JSON está mal formado (comas extra, etc.), limpiamos el manualmente
            limpio = re.sub(r'["\'{}]', '', json_blob)
            limpio = re.sub(r'pregunta_del_paciente\s*:\s*', '', limpio, flags=re.IGNORECASE)
            return limpio.strip()
            
    # Si no hay llaves (JSON), limpiamos Markdown y devolvemos el texto limpio
    texto_limpio = re.sub(r'```[a-z]*|```|\*\*', '', texto_str).strip()
    
    if len(texto_limpio) > 400:
        return texto_limpio[:200] + "..."
        
    return texto_limpio


df_raw = pd.read_csv("resultados_etiquetado_12_topicos_5modelos.csv")


df_orig = pd.DataFrame()
df_orig['cluster_id'] = df_raw['cluster']


modelos = {
    'res_qwen3_30b': 'qwen3:30b',
    'res_deepseek-r1_32b': 'deepseek-r1:32b',
    'res_gemma3_27b': 'gemma3:27b',
    'res_gpt-oss_120b' : 'gpt-oss:120b',
    'res_nemotron-3-nano_30b' : 'nemotron-3-nano:30b'
}

print("Iniciando limpieza de respuestas...")
for col_csv, nombre_final in modelos.items():
    if col_csv in df_raw.columns:
        print(f"  > Procesando columna: {col_csv}")
        df_orig[nombre_final] = df_raw[col_csv].apply(limpiar_y_extraer_json_robusto)

df_orig.set_index('cluster_id', inplace=True)
df_orig.to_csv("preguntas_originales_v4_5modelos.csv")

print("\n¡Limpieza completada!")
print(f"Archivo guardado como: preguntas_originales_v4_5modelos.csv")
print(df_orig.head())

Iniciando limpieza de respuestas...
  > Procesando columna: res_qwen3_30b
  > Procesando columna: res_deepseek-r1_32b
  > Procesando columna: res_gemma3_27b
  > Procesando columna: res_gpt-oss_120b
  > Procesando columna: res_nemotron-3-nano_30b

¡Limpieza completada!
Archivo guardado como: preguntas_originales_v4_5modelos.csv
                                                    qwen3:30b  \
cluster_id                                                      
0                       ¿Este medicamento es exento de sodio?   
1           ¿Puedo tomar este medicamento si estoy embaraz...   
2                              ¿Qué hago si olvido una dosis?   
3           ¿Este medicamento puede causar erupciones en l...   
4           ¿Qué efectos secundarios puede causar este med...   

                                              deepseek-r1:32b  \
cluster_id                                                      
0           ¿Cuál es la dosis correcta para este medicamen...   
1           ¿Cuáles 